In [43]:
import subprocess
import shutil
import sys
import re

In [27]:
def _correr(comando):
    """Ejecuta un comando de shell silenciando la salida; retorna True si OK."""
    resultado = subprocess.run(comando, shell=True,
                                stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL)
    return resultado.returncode == 0


def instalar_dependencias():
    """
    Revisa si están instalados pdflatex, graphviz (binario) y las librerías
    de Python necesarias (pylatex, graphviz). Si falta algo, lo instala.
    Diseñado para correr como root en Google Colab (usa apt-get sin sudo).
    """
    print(">> Verificando dependencias...")

    # --- 1a. LaTeX (pdflatex) ---
    if shutil.which("pdflatex") is None:
        print("   Instalando LaTeX (esto puede tardar unos minutos)...")
        paquetes = ("lmodern texlive-latex-base texlive-latex-recommended "
                    "texlive-latex-extra texlive-fonts-recommended "
                    "texlive-lang-spanish")
        ok = _correr(f"apt-get -qq update && apt-get -qq install -y {paquetes}")
        if not ok:
            # Reintento sin el paquete de idioma español, por si el nombre
            # del paquete cambia en algunas imágenes de Colab.
            print("   Reintentando instalación sin texlive-lang-spanish...")
            paquetes_alt = ("lmodern texlive-latex-base texlive-latex-recommended "
                             "texlive-latex-extra texlive-fonts-recommended")
            _correr(f"apt-get -qq install -y {paquetes_alt}")
    else:
        print("   LaTeX ya está instalado. OK.")

    # --- 1b. Graphviz (binario del sistema, para dibujar el flujograma) ---
    if shutil.which("dot") is None:
        print("   Instalando Graphviz...")
        _correr("apt-get -qq install -y graphviz")
    else:
        print("   Graphviz ya está instalado. OK.")

    # --- 1c. Librerías de Python ---
    try:
        import pylatex  # noqa: F401
    except ImportError:
        print("   Instalando PyLaTeX...")
        _correr(f"{sys.executable} -m pip install --quiet pylatex")

    try:
        import graphviz  # noqa: F401
    except ImportError:
        print("   Instalando el paquete graphviz de Python...")
        _correr(f"{sys.executable} -m pip install --quiet graphviz")

    print(">> Dependencias listas.\n")


# Instalamos ANTES de importar pylatex/graphviz, para que funcione incluso
# en una máquina nueva de Colab que no los tenga.
instalar_dependencias()

from pylatex import (Document, Section, Subsection, Package, Figure,      # noqa: E402
                      NewPage, Itemize, Center, Command)
from pylatex.utils import NoEscape, escape_latex                          # noqa: E402
import graphviz                                                            # noqa: E402

>> Verificando dependencias...
   LaTeX ya está instalado. OK.
   Graphviz ya está instalado. OK.
   Instalando PyLaTeX...
>> Dependencias listas.



In [28]:

TITULO_PROYECTO = "ChatMP3: Chatbot Básico Recomendador de Música"
AUTORES = ["Nombre Apellido 1", "Nombre Apellido 2"]        # <-- EDITAR
UNIVERSIDAD = "Universidad Nacional de Colombia"             # <-- EDITAR
CURSO = "Nombre del curso"                                   # <-- EDITAR
PROFESOR = "Nombre del profesor"                              # <-- EDITAR
CIUDAD_FECHA = "Bogotá D.C., 2026"                            # <-- EDITAR
REPOSITORIO_GITHUB = "github.com/danielmogo25/Proyecto-UNAL-ChatMP3"

NOMBRE_PDF = "Informe_ChatMP3"          # nombre del archivo final (sin .pdf)

# Rutas donde el script buscará los archivos del proyecto. Si el archivo
# exacto no existe, se intenta encontrar automáticamente uno parecido
# (glob) en la carpeta actual, por si Colab le cambió el nombre al subirlo.
RUTA_CSV_PREFERIDA = "prueba - Hoja 1.csv"
RUTA_DB_PREFERIDA = "base_datos_chat_mp3(2).sqlite"
CARPETA_CAPTURAS = "capturas"

## 3. Utilidades para escribir texto en LaTeX sin dolores de cabeza
Funciones auxiliares para escapar texto, agregar párrafos, listas, bloques
de código y cajas de nota dentro del documento LaTeX.

In [29]:
#   - Los acentos y la "ñ" se manejan solos gracias a inputenc/fontenc + babel.
#   - Los caracteres especiales de LaTeX (%, _, &, #, etc.) se escapan solos.
#   - Para resaltar código dentro de una frase, se usa `comillas_invertidas`,
#     igual que en Markdown, y se convierte automáticamente a \texttt{...}.

def formatear_texto(texto):
    """Convierte texto con `código` estilo Markdown a LaTeX seguro."""
    partes = re.split(r'(`[^`]+`)', texto)
    contenido = ''
    for parte in partes:
        if parte.startswith('`') and parte.endswith('`'):
            contenido += r'\texttt{' + escape_latex(parte[1:-1]) + '}'
        else:
            contenido += escape_latex(parte)
    return contenido


def parrafo(doc, texto):
    """Agrega un párrafo normal (con salto de línea al final)."""
    doc.append(NoEscape(formatear_texto(texto) + r'\par\vspace{6pt}'))


def lista(doc, items):
    """Agrega una lista con viñetas a partir de una lista de strings."""
    with doc.create(Itemize()) as it:
        for item in items:
            it.add_item(NoEscape(formatear_texto(item)))


def bloque_codigo(doc, codigo, lenguaje="Python"):
    """Agrega un bloque de código con resaltado (paquete listings)."""
    codigo = codigo.replace('\t', '    ')
    bloque = ("\\begin{lstlisting}[language=" + lenguaje + "]\n"
              + codigo + "\n\\end{lstlisting}")
    doc.append(NoEscape(bloque))


def caja_nota(doc, titulo, texto, color="blue!5", color_borde="blue!40"):
    """Agrega una caja de color con un título y un texto (paquete tcolorbox)."""
    contenido = formatear_texto(texto)
    bloque = (r'\begin{tcolorbox}[colback=' + color + r',colframe=' + color_borde
              + r',title=' + formatear_texto(titulo) + r']' + '\n'
              + contenido + '\n' + r'\end{tcolorbox}')
    doc.append(NoEscape(bloque))

## 5. Generación del flujograma (Graphviz)
Dibuja el flujograma del ciclo principal del chatbot (`main_chat()`).

In [30]:
def generar_flujograma(ruta_salida="flujograma"):
    """
    Dibuja el flujograma real del chatbot (basado en la función main_chat())
    y lo guarda como PNG. Devuelve la ruta del archivo generado.
    """
    dot = graphviz.Digraph('flujo_chatmp3', format='png')
    dot.attr(rankdir='TB', bgcolor='white', nodesep='0.5', ranksep='0.6',
             splines='polyline')
    dot.attr('node', fontname='Helvetica', fontsize='11', margin='0.15,0.1')
    dot.attr('edge', fontname='Helvetica', fontsize='9')

    C_START, C_END = '#a9d6b5', '#e8a6a1'
    C_BOX, C_DEC, C_PREP = '#eef2f7', '#ffe08a', '#e0e0e0'

    dot.node('prep', 'Fase previa (una sola vez)\ninsertar_cancion():\n'
                     'lee el CSV y consulta la API\nde iTunes para llenar la BD SQLite',
             shape='box', style='rounded,filled,dashed', fillcolor=C_PREP)
    dot.node('inicio', 'Inicio\n(main_chat)', shape='oval', style='filled',
             fillcolor=C_START)
    dot.node('saludo', 'Saluda y pregunta\nel nombre del usuario',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('pedir_opcion', 'Pregunta: ¿artista,\npreguntas o salir?',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('decision', '¿Qué palabra clave\ncontiene la respuesta?',
             shape='diamond', style='filled', fillcolor=C_DEC)

    dot.node('art1', 'guardar_artista():\npide nombre del artista',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('art2', 'buscador_por_termino():\nconsulta en vivo la API iTunes',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('art3', 'Muestra portada, datos\ny audio de preview',
             shape='box', style='rounded,filled', fillcolor=C_BOX)

    dot.node('pref1', 'preguntas_gustos():\ngénero, duración y década',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('pref2', 'query_base_random():\nfiltra la BD SQLite local',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('pref3', 'mostrar_cancion():\nelige una al azar y la muestra',
             shape='box', style='rounded,filled', fillcolor=C_BOX)

    dot.node('error', 'Mensaje de error:\nopción no reconocida',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('salir', 'Mensaje de despedida',
             shape='box', style='rounded,filled', fillcolor=C_BOX)
    dot.node('fin', 'Fin', shape='oval', style='filled', fillcolor=C_END)

    dot.edge('prep', 'inicio', style='dashed', label='opcional / previo')
    dot.edge('inicio', 'saludo')
    dot.edge('saludo', 'pedir_opcion')
    dot.edge('pedir_opcion', 'decision')

    dot.edge('decision', 'art1', label='artista/cantante/1/uno')
    dot.edge('decision', 'pref1', label='pregunta/gustos/2/dos')
    dot.edge('decision', 'salir', label='salir/cerrar/3/tres')
    dot.edge('decision', 'error', label='no reconocida')

    dot.edge('art1', 'art2')
    dot.edge('art2', 'art3')
    dot.edge('pref1', 'pref2')
    dot.edge('pref2', 'pref3')

    dot.edge('art3', 'pedir_opcion', constraint='false', label='vuelve al menú')
    dot.edge('pref3', 'pedir_opcion', constraint='false', label='vuelve al menú')
    dot.edge('error', 'pedir_opcion', constraint='false')
    dot.edge('salir', 'fin')

    dot.render(ruta_salida, cleanup=True)
    return ruta_salida + ".png"

## 6. Transcripción real de una ejecución del chatbot
Conversación real usada como evidencia en la sección de demostración.

In [31]:
# Esta es una conversación real, tomada de una ejecución del notebook
# original del proyecto (main_chat()), que se usa como evidencia de
# funcionamiento en la sección de demostración.

TRANSCRIPCION_DEMO = """\
Hola, bienvenido a ChatMP3! Estoy para ayudarte a encontrar tus nuevas \
canciones favoritas
¿Cuál es tu nombre? juan
Puedo buscar canciones en base preguntas generales sobre tus gustos \
musicales o con base em un artista especifico

Dime juan, cual opcion deseas (por artista o preguntas) o tambien si \
deseas salir del programa:
preguntas
Entendido, recomendare con preguntas sobre tus gustos musicales

Vale, como primera pregunta entre pop, rock, rap o electrónica cual es \
tu genero favorito?
rap
Como segunda pregunta entre que duración es tu preferida de las \
siguientes: menos de 3 min, entre 3 y 4, o 4 o mas
mas
Como tercera pregunta, ¿cual decada de musica prefieres: los 90, los \
2000 o del 2010 en adelante?
2000
==================================================
TITULO: I Wonder
ALBUM: Graduation
ARTISTA: Kanye West
GENERO: Hip-Hop/Rap
DURACION: 4:03
AÑO SALIDA: 2007
[Se muestra la carátula del álbum y se reproduce un audio de preview]

Dime juan, cual opcion deseas (por artista o preguntas) o tambien si \
deseas salir del programa:
artista
Entendido, recomendare a un artista en especifico

Dime el nombre del artista que quieres escuchar: Michael Jackson
CODIGO DE ESTADO 200
==================================================
TITULO: Beat It
ALBUM: Thriller
ARTISTA: Michael Jackson
GENERO: Pop
DURACION: 4:18
AÑO SALIDA: 1982
[Se muestra la carátula del álbum y se reproduce un audio de preview]

Dime juan, cual opcion deseas (por artista o preguntas) o tambien si \
deseas salir del programa:
salir
Entendido, saliendo del programa, hasta luego!
"""

## 7. Construcción de cada sección del informe

### 7.1 Portada

In [32]:
def construir_portada(doc):
    doc.append(NoEscape(r'\begin{titlepage}'))
    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'\Large ' + formatear_texto(UNIVERSIDAD) + r' \\[0.3cm]'))
    doc.append(NoEscape(r'\large ' + formatear_texto(CURSO) + r' \\[2.5cm]'))
    doc.append(NoEscape(r'\rule{\linewidth}{0.5mm} \\[0.4cm]'))
    doc.append(NoEscape(r'{\huge \bfseries ' + formatear_texto(TITULO_PROYECTO) + r'} \\[0.4cm]'))
    doc.append(NoEscape(r'\rule{\linewidth}{0.5mm} \\[2cm]'))
    doc.append(NoEscape(r'\large \textbf{Informe del proyecto}\\[2cm]'))
    autores_tex = r' \\ '.join(formatear_texto(a) for a in AUTORES)
    doc.append(NoEscape(r'\large ' + autores_tex + r'\\[1cm]'))
    doc.append(NoEscape(r'\normalsize Profesor(a): ' + formatear_texto(PROFESOR) + r'\\[2cm]'))
    doc.append(NoEscape(r'\vfill'))
    doc.append(NoEscape(r'\large ' + formatear_texto(CIUDAD_FECHA)))
    doc.append(NoEscape(r'\end{center}'))
    doc.append(NoEscape(r'\end{titlepage}'))

### 7.2 Introducción

In [47]:
def construir_introduccion(doc):
    with doc.create(Section('Introducción')):
        parrafo(doc,
            f"Este informe documenta el desarrollo de `{TITULO_PROYECTO}`, un "
            "proyecto que consiste en un chatbot de consola, ejecutado en Google "
            "Colab, capaz de recomendar canciones a un usuario. El chatbot "
            "conversa en español, hace preguntas sencillas sobre los gustos "
            "musicales de la persona (o recibe el nombre de un artista) y, con "
            "base en esa información, sugiere una canción junto con su carátula "
            "y un fragmento de audio de muestra.")
        parrafo(doc,
            "El proyecto combina varias herramientas típicas de un programa de "
            "consulta e interacción con datos: consumo de una API externa "
            "(iTunes Search API), almacenamiento persistente en una base de "
            "datos SQLite, procesamiento de un archivo CSV como fuente inicial "
            "de canciones, y un ciclo de conversación controlado completamente "
            "por funciones de Python (`input()` / `print()`), sin usar "
            "modelos de lenguaje ni procesamiento de lenguaje natural "
            "avanzado.")
        parrafo(doc,
            "En las siguientes secciones se explica el problema que motivó el "
            "proyecto, el algoritmo de recomendación utilizado, un flujograma "
            "del programa, una explicación detallada del código, una "
            "demostración de su funcionamiento con un caso real, capturas de "
            "pantalla y las instrucciones para ejecutarlo.")

### 7.3 Descripción del problema

In [34]:
def construir_problema(doc):
    with doc.create(Section('Descripción del problema')):
        parrafo(doc,
            "Encontrar música nueva que se ajuste a los gustos de una persona "
            "puede ser una tarea tediosa: los catálogos de las plataformas de "
            "streaming son enormes, y sus algoritmos de recomendación suelen "
            "ser complejos, cerrados y difíciles de entender o de replicar "
            "como ejercicio académico. Además, muchas personas simplemente "
            "quieren una sugerencia rápida y puntual (\"quiero algo de rock de "
            "los 2000\", \"ponme algo de tal artista\") sin tener que navegar "
            "por menús ni crear una cuenta en un servicio externo.")
        parrafo(doc,
            "El problema que aborda este proyecto es, entonces, construir un "
            "sistema simple, conversacional y en español, que permita a un "
            "usuario obtener una recomendación de canción de dos formas "
            "distintas:")
        lista(doc, [
            "Indicando el nombre de un artista o cantante específico que "
            "quiere escuchar.",
            "Respondiendo tres preguntas sencillas sobre sus preferencias "
            "generales: género musical, duración aproximada de la canción y "
            "década de lanzamiento.",
        ])
        parrafo(doc,
            "Adicionalmente, el proyecto busca resolver un segundo problema más "
            "técnico: cómo construir y mantener una base de datos musical "
            "propia (con título, artista, álbum, género, duración, año, "
            "carátula y enlace de audio) a partir de una simple lista de "
            "nombres de canciones, sin tener que digitar manualmente toda esa "
            "información, delegando esa tarea a una API pública de música "
            "(iTunes Search API).")

### 7.4 Descripción del algoritmo solución

In [49]:
def construir_algoritmo(doc):
    with doc.create(Section('Descripción del algoritmo solución')):
        parrafo(doc,
            "La solución implementada es un sistema basado en reglas y "
            "filtrado de criterios (no un modelo de aprendizaje automático ni "
            "un sistema de recomendación colaborativo). Puede describirse en "
            "dos grandes fases:")

        with doc.create(Subsection('Fase 1: Construcción de la base de datos')):
            parrafo(doc,
                "Antes de poder recomendar canciones, el programa necesita una "
                "base de datos local con información musical. Para ello:")
            lista(doc, [
                "Se parte de un archivo CSV con una lista de títulos de "
                "canciones curada manualmente (una por fila).",
                "Por cada título, se consulta la API pública de iTunes "
                "(`itunes.apple.com/search`) pidiendo la primera coincidencia "
                "de tipo canción.",
                "De la respuesta en formato JSON se extraen: título, álbum, "
                "artista, género, año de lanzamiento, duración (convertida de "
                "milisegundos a formato `minutos:segundos`), la URL de un "
                "fragmento de audio (`previewUrl`) y la URL de la carátula "
                "(`artworkUrl100`).",
                "Estos datos se insertan en dos tablas de una base de datos "
                "SQLite: `Track` (canciones) y `Genero` (catálogo de "
                "géneros), evitando duplicados si la canción ya existe.",
                "Este proceso se limita a 100 canciones por ejecución para no "
                "saturar la API externa, y omite canciones ya presentes en la "
                "base de datos para no repetir consultas innecesarias.",
            ])

        with doc.create(Subsection('Fase 2: Recomendación conversacional')):
            parrafo(doc,
                "Una vez existe la base de datos, el chatbot ofrece dos "
                "estrategias de recomendación, seleccionadas por el usuario "
                "mediante palabras clave (sin usar NLP): ")
            lista(doc, [
                "`Búsqueda por artista`: el usuario escribe el nombre de un "
                "artista; el programa consulta la API de iTunes en tiempo "
                "real (no la base de datos local) y muestra la primera "
                "canción encontrada de ese artista, con su carátula y un "
                "fragmento de audio.",
                "`Búsqueda por preferencias`: el programa hace tres preguntas "
                "de opción fija (género, duración, década). Cada respuesta se "
                "valida buscando palabras clave esperadas (por ejemplo "
                "`rock`, `rap`, `pop`, `electronica`); si no reconoce ninguna "
                "palabra clave válida, o reconoce más de una a la vez, vuelve "
                "a preguntar. Una vez tiene las tres respuestas válidas, "
                "traduce cada una a un criterio de búsqueda SQL: el género "
                "elegido se expande a un grupo de géneros relacionados de "
                "iTunes (por ejemplo `rock` incluye \"Rock\", \"Hard Rock\", "
                "\"Indie Rock\", \"Rock y Alternativo\" y \"Metal\"), la "
                "duración se traduce a un rango de segundos, y la década a un "
                "rango de años. Con esos tres criterios se filtra la tabla "
                "`Track` de la base de datos local, y de los resultados que "
                "cumplen todos los criterios se elige uno al azar.",
            ])
            parrafo(doc,
                "En ambos casos, la canción elegida se presenta al usuario "
                "mostrando su carátula (descargada e incrustada con "
                "`IPython.display.Image`), sus datos principales en texto, y "
                "un reproductor de audio embebido (`IPython.display.Audio`) "
                "con el fragmento de previsualización.")

### 7.5 Flujograma (sección del informe)

In [36]:
def construir_flujograma(doc, ruta_imagen):
    with doc.create(Section('Flujograma')):
        parrafo(doc,
            "El siguiente flujograma resume el ciclo principal del chatbot, "
            "implementado en la función `main_chat()`, incluyendo la fase "
            "previa (fuera de línea) de carga de la base de datos y las dos "
            "rutas de recomendación (por artista o por preguntas):")
        with doc.create(Figure(position='h!')) as fig:
            fig.add_image(ruta_imagen, width=NoEscape(r'0.92\textwidth'))
            fig.add_caption(NoEscape(formatear_texto(
                "Flujograma del ciclo conversacional de ChatMP3")))

### 7.6 Función auxiliar para explicar bloques de código

In [37]:
def _codigo_explicacion(doc, titulo, explicacion, codigo):
    with doc.create(Subsection(titulo)):
        parrafo(doc, explicacion)
        bloque_codigo(doc, codigo)

### 7.7 Explicación del código

In [38]:
def construir_explicacion_codigo(doc):
    with doc.create(Section('Explicación del código')):
        parrafo(doc,
            "A continuación se explica el propósito de cada bloque del "
            "código, agrupado en cuatro partes: configuración inicial y base "
            "de datos, conexión con la API de iTunes, carga de canciones a la "
            "base de datos, consulta/presentación de recomendaciones y flujo "
            "conversacional.")

        _codigo_explicacion(
            doc, "Configuración inicial y creación de la base de datos",
            "Se importan las librerías necesarias (`requests` para consumir "
            "la API, `sqlite3` para la base de datos, `csv` para leer el "
            "archivo fuente, `IPython.display` para mostrar imágenes y audio "
            "en el notebook, entre otras). Luego se crea, si no existe, la "
            "base de datos SQLite con dos tablas: `Track`, que guarda cada "
            "canción con su artista, álbum, género, duración, año, enlace de "
            "audio y de carátula; y `Genero`, un catálogo de géneros "
            "musicales únicos.",
            "import requests\n"
            "import sqlite3\n"
            "import csv\n"
            "from IPython.display import Audio, Image, display\n\n"
            "conn = sqlite3.connect('base_datos_chat_mp3(2).sqlite')\n"
            "cur = conn.cursor()\n"
            "cur.executescript('''\n"
            "CREATE TABLE IF NOT EXISTS Track(\n"
            "  id INTEGER PRIMARY KEY AUTOINCREMENT UNIQUE,\n"
            "  titulo TEXT UNIQUE, artista INTEGER, album INTEGER,\n"
            "  genero INTEGER, duracion INTEGER, fecha INTEGER,\n"
            "  preview TEXT, imagen TEXT\n"
            ");\n"
            "CREATE TABLE IF NOT EXISTS Genero(\n"
            "  id INTEGER PRIMARY KEY AUTOINCREMENT UNIQUE,\n"
            "  nombre TEXT UNIQUE\n"
            ")\n"
            "''')\n"
            "conn.commit()")

        _codigo_explicacion(
            doc, "Conexión con la API de iTunes: retorna_metacancion()",
            "Esta función recibe un término de búsqueda (por ejemplo el "
            "título de una canción) y consulta el endpoint público "
            "`itunes.apple.com/search`, pidiendo un único resultado de tipo "
            "canción (`entity=song, limit=1`). De la respuesta en JSON extrae "
            "título, álbum, artista, año de lanzamiento, género, y convierte "
            "la duración de milisegundos a formato `minutos:segundos`. "
            "Devuelve `None` si la API no encuentra resultados.",
            "def retorna_metacancion(termino):\n"
            "    url = \"https://itunes.apple.com/search\"\n"
            "    params = {\"term\": termino, \"media\": \"music\",\n"
            "              \"entity\": \"song\", \"limit\": 1}\n"
            "    respuesta = requests.get(url, params=params, timeout=10)\n"
            "    resultados = respuesta.json()[\"results\"]\n"
            "    if len(resultados) == 0:\n"
            "        return None\n"
            "    dato = resultados[0]\n"
            "    titulo = dato.get('trackName')\n"
            "    album = dato.get(\"collectionName\")\n"
            "    artista = dato.get('artistName')\n"
            "    preview = dato.get('previewUrl')\n"
            "    fecha = int(dato.get('releaseDate', '0000')[:4])\n"
            "    genero = dato.get('primaryGenreName')\n"
            "    milisegundos = dato.get('trackTimeMillis')\n"
            "    imagen = dato.get('artworkUrl100')\n"
            "    if milisegundos:\n"
            "        minutos = milisegundos // 60000\n"
            "        segundos = (milisegundos % 60000) // 1000\n"
            "        duracion = f\"{minutos}:{segundos:02d}\"\n"
            "    else:\n"
            "        duracion = None\n"
            "    return [titulo, duracion, album, artista, fecha,\n"
            "            genero, preview, imagen]")

        _codigo_explicacion(
            doc, "Carga masiva de canciones: insertar_cancion()",
            "Lee el archivo CSV línea por línea, usando solo la primera "
            "columna (el título de la canción) como término de búsqueda. Si "
            "el título ya existe en la tabla `Track`, lo omite para no "
            "repetir consultas a la API. Para cada canción nueva, llama a "
            "`retorna_metacancion()`, inserta el género (si es nuevo) en la "
            "tabla `Genero`, y luego inserta la canción completa en la tabla "
            "`Track`. Por control, detiene el proceso después de 100 "
            "canciones por ejecución.",
            "def insertar_cancion(nombre_csv):\n"
            "    conn = sqlite3.connect('base_datos_chat_mp3(2).sqlite')\n"
            "    cur = conn.cursor()\n"
            "    count = 0\n"
            "    with open(nombre_csv, \"r\", encoding=\"utf-8\") as f:\n"
            "        lector = csv.reader(f)\n"
            "        for fila in lector:\n"
            "            if count > 100:\n"
            "                break\n"
            "            termino = fila[0].strip()\n"
            "            if not termino:\n"
            "                continue\n"
            "            cur.execute(\"SELECT id FROM Track WHERE titulo = ?\",\n"
            "                        (termino,))\n"
            "            if cur.fetchone() is not None:\n"
            "                continue\n"
            "            pieces = retorna_metacancion(termino)\n"
            "            count += 1\n"
            "            if pieces is None:\n"
            "                continue\n"
            "            (titulo, duracion, album, artista, fecha,\n"
            "             genero, preview, imagen) = pieces\n"
            "            cur.execute(\"INSERT OR IGNORE INTO Genero (nombre) \"\n"
            "                        \"VALUES (?)\", (genero,))\n"
            "            cur.execute('''INSERT OR IGNORE INTO Track\n"
            "                (titulo, artista, album, duracion, fecha,\n"
            "                 preview, imagen, genero)\n"
            "                VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',\n"
            "                (titulo, artista, album, duracion, fecha,\n"
            "                 preview, imagen, genero))\n"
            "            conn.commit()\n"
            "    conn.close()")

        _codigo_explicacion(
            doc, "Consulta local por criterios: query_base_random()",
            "Traduce las tres preferencias del usuario (género, duración, "
            "década, todas como texto) a criterios concretos de una consulta "
            "SQL: el género se expande a una tupla de géneros afines de "
            "iTunes; la duración a un rango de segundos calculado a partir "
            "del texto `minutos:segundos` guardado en la base de datos "
            "(usando funciones de cadenas de SQLite como `substr` e "
            "`instr`); y la década a un rango de años. Devuelve la lista "
            "completa de canciones que cumplen las tres condiciones a la "
            "vez.",
            "def query_base_random(genero, duracion, decada):\n"
            "    conn = sqlite3.connect('base_datos_chat_mp3(2).sqlite')\n"
            "    cur = conn.cursor()\n"
            "    if genero == \"rock\":\n"
            "        parametros_genero = (\"Indie Rock\", \"Hard Rock\", \"Rock\",\n"
            "                              \"Rock y Alternativo\", \"Metal\")\n"
            "    # ... (mapeos similares para pop, rap, electronica)\n"
            "    if duracion == \"menos\":\n"
            "        p_dur1, p_dur2 = 0, 180\n"
            "    # ... (mapeos similares para 'entre' y 'mas')\n"
            "    cur.execute(f'''\n"
            "        SELECT * FROM Track\n"
            "        WHERE genero IN {parametros_genero}\n"
            "        AND (fecha BETWEEN {p_decada1} AND {p_decada2})\n"
            "        AND (CAST(substr(duracion, 1,\n"
            "                instr(duracion, ':') - 1) AS INTEGER) * 60\n"
            "             + CAST(substr(duracion,\n"
            "                instr(duracion, ':') + 1) AS INTEGER))\n"
            "             BETWEEN {p_dur1} AND {p_dur2}\n"
            "    ''')\n"
            "    return cur.fetchall()")

        _codigo_explicacion(
            doc, "Presentación de resultados: mostrar_cancion() y descargar_imagen()",
            "`mostrar_cancion()` recibe la lista de canciones que cumplen los "
            "criterios, elige una al azar con `random.shuffle`, descarga su "
            "carátula con `descargar_imagen()` (una función auxiliar que "
            "guarda en disco cualquier archivo binario a partir de una URL, "
            "manejando tiempos de espera y errores de red), imprime los "
            "datos de la canción y muestra tanto la imagen como un "
            "reproductor de audio con el fragmento de previsualización "
            "descargado en el momento.",
            "def descargar_imagen(url, nombre_archivo):\n"
            "    respuesta = requests.get(url, stream=True, timeout=10)\n"
            "    respuesta.raise_for_status()\n"
            "    with open(nombre_archivo, \"wb\") as archivo:\n"
            "        for bloque in respuesta.iter_content(chunk_size=8192):\n"
            "            archivo.write(bloque)\n"
            "    return True\n\n"
            "def mostrar_cancion(lista_canciones):\n"
            "    lista_tuplas = list(lista_canciones)\n"
            "    if len(lista_tuplas) == 0:\n"
            "        print(\"No se encontró una canción como la que buscas\")\n"
            "        return False\n"
            "    random.shuffle(lista_tuplas)\n"
            "    (id, titulo, artista, album, genero, duracion,\n"
            "     fecha, preview, imagen) = list(lista_tuplas[0])\n"
            "    archivo_local = f\"{titulo}.jpg\"\n"
            "    descargar_imagen(imagen, archivo_local)\n"
            "    print(f\"TITULO:{titulo}\\nARTISTA:{artista}...\")\n"
            "    display(Image(archivo_local))\n"
            "    info = requests.get(preview, timeout=10)\n"
            "    with open(f\"{titulo}Preview.m4a\", \"wb\") as archi:\n"
            "        archi.write(info.content)\n"
            "    display(Audio(f\"{titulo}Preview.m4a\"))\n"
            "    return True")

        _codigo_explicacion(
            doc, "Flujo conversacional: guardar_artista(), preguntas_gustos() "
                 "y main_chat()",
            "`guardar_artista()` pide el nombre de un artista y llama a "
            "`buscador_por_termino()` (consulta directa a la API de iTunes, "
            "sin pasar por la base de datos local). `preguntas_gustos()` "
            "hace las tres preguntas de preferencias, validando cada "
            "respuesta con listas de palabras clave (`generos`, "
            "`claves_duraciones`, `claves_decadas`) hasta obtener una "
            "respuesta reconocible, y al final llama a `query_base_random()` "
            "seguido de `mostrar_cancion()`. Finalmente, `main_chat()` es la "
            "función principal: saluda, pide el nombre del usuario y entra "
            "en un bucle infinito que pregunta qué opción desea (artista, "
            "preguntas o salir), interpretando la respuesta por palabras "
            "clave y dirigiendo el flujo a la función correspondiente, hasta "
            "que el usuario decide salir.",
            "def main_chat():\n"
            "    print(\"Hola, bienvenido a ChatMP3!...\")\n"
            "    nombre = input(\"¿Cuál es tu nombre? \").strip()\n"
            "    while True:\n"
            "        respuesta = input(f\"Dime {nombre}, cual opcion "
            "deseas...\")\n"
            "        if \"artista\" in respuesta or \"1\" in respuesta:\n"
            "            guardar_artista(respuesta)\n"
            "            continue\n"
            "        elif \"pregunta\" in respuesta or \"2\" in respuesta:\n"
            "            preguntas_gustos()\n"
            "            continue\n"
            "        elif \"salir\" in respuesta or \"3\" in respuesta:\n"
            "            print(\"Entendido, saliendo del programa...\")\n"
            "            return\n"
            "        else:\n"
            "            print(\"Perdon, no pude entender...\")\n\n"
            "main_chat()")

### 7.8 Demostración de funcionalidades

In [39]:
def construir_demostracion(doc, cancion_ejemplo, ruta_imagen_ejemplo):
    with doc.create(Section('Demostración de funcionalidades')):
        parrafo(doc,
            "A continuación se muestra una conversación real, registrada "
            "durante una ejecución del chatbot en Google Colab, donde el "
            "usuario prueba las dos formas de recomendación: primero por "
            "preferencias (género, duración, década) y luego por artista "
            "específico, y finalmente sale del programa.")
        bloque_codigo(doc, TRANSCRIPCION_DEMO, lenguaje="")
        parrafo(doc,
            "Como se observa, ante la búsqueda por preferencias "
            "(`rap`, `mas` de 4 minutos, década `2000`) el sistema recomendó "
            "la canción `I Wonder` de Kanye West (2007), y ante la búsqueda "
            "directa por el artista `Michael Jackson` recomendó la canción "
            "`Beat It` (1982). En ambos casos el programa mostró en pantalla "
            "la carátula del álbum correspondiente y reprodujo un fragmento "
            "de audio de previsualización.")

        if cancion_ejemplo and ruta_imagen_ejemplo and os.path.isfile(ruta_imagen_ejemplo):
            parrafo(doc,
                "A modo de ejemplo adicional, esta es una carátula real "
                "obtenida en el momento de generar este informe, "
                f"correspondiente a la canción `{cancion_ejemplo['titulo']}` "
                f"de `{cancion_ejemplo['artista']}` "
                f"({cancion_ejemplo['fecha']}, género "
                f"`{cancion_ejemplo['genero']}`, duración "
                f"`{cancion_ejemplo['duracion']}`), tal como la mostraría "
                "`mostrar_cancion()` en Colab:")
            with doc.create(Figure(position='h!')) as fig:
                fig.add_image(ruta_imagen_ejemplo, width=NoEscape(r'0.28\textwidth'))
                fig.add_caption(NoEscape(formatear_texto(
                    f"Carátula real de \"{cancion_ejemplo['titulo']}\" - "
                    f"{cancion_ejemplo['artista']}, obtenida de la base de "
                    "datos del proyecto")))
        else:
            caja_nota(doc, "Nota",
                "No se encontró la base de datos SQLite del proyecto en la "
                "carpeta actual, así que esta sección omite una carátula de "
                "ejemplo adicional. Si colocas el archivo "
                f"`{RUTA_DB_PREFERIDA}` junto a este script y lo vuelves a "
                "ejecutar, se incluirá automáticamente una carátula real "
                "descargada de tu propia base de datos.",
                color="yellow!8", color_borde="yellow!60!black")

### 7.9 Capturas de pantalla

In [40]:
def construir_capturas(doc, capturas):
    with doc.create(Section('Capturas de pantalla')):
        if capturas:
            parrafo(doc,
                "A continuación se presentan capturas de pantalla reales del "
                "chatbot funcionando en Google Colab:")
            for i, ruta in enumerate(capturas, start=1):
                nombre = os.path.splitext(os.path.basename(ruta))[0]
                nombre = nombre.replace('_', ' ').replace('-', ' ')
                with doc.create(Figure(position='h!')) as fig:
                    fig.add_image(ruta, width=NoEscape(r'0.85\textwidth'))
                    fig.add_caption(NoEscape(formatear_texto(
                        f"Captura {i}: {nombre}")))
        else:
            parrafo(doc,
                "Esta sección se completa automáticamente con tus propias "
                "capturas de pantalla del chatbot en funcionamiento.")
            caja_nota(doc, "Cómo agregar tus capturas de pantalla",
                "1) Ejecuta el chatbot (el notebook original) en Google Colab "
                "y toma capturas de pantalla de la conversación, las "
                "carátulas y el reproductor de audio. "
                f"2) Crea, junto a este script, una carpeta llamada "
                f"`{CARPETA_CAPTURAS}` y guarda ahí tus imágenes (.png o "
                ".jpg). 3) Vuelve a ejecutar este script: las imágenes se "
                "incluirán automáticamente en esta sección, una por página, "
                "en orden alfabético según el nombre del archivo.",
                color="yellow!8", color_borde="yellow!60!black")

### 7.10 Instrucciones de ejecución

In [41]:
def construir_instrucciones(doc):
    with doc.create(Section('Instrucciones de ejecución')):
        with doc.create(Subsection('Requisitos')):
            lista(doc, [
                "Una cuenta de Google y acceso a Google Colab "
                "(`colab.research.google.com`).",
                "Conexión a internet (el programa consulta la API pública de "
                "iTunes en tiempo real).",
                f"Los archivos del proyecto: el notebook original, el CSV de "
                f"canciones y la base de datos SQLite (repositorio: "
                f"`{REPOSITORIO_GITHUB}`).",
            ])
        with doc.create(Subsection('Pasos para ejecutar el chatbot')):
            with doc.create(Itemize()) as it:
                it.add_item(NoEscape(formatear_texto(
                    "Abre el notebook `Chat_mp3.ipynb` en Google Colab "
                    "(directamente desde GitHub o subiéndolo manualmente).")))
                it.add_item(NoEscape(formatear_texto(
                    "Ejecuta en orden las celdas de importación de librerías "
                    "y creación de la base de datos.")))
                it.add_item(NoEscape(formatear_texto(
                    "Si es la primera vez, sube el archivo CSV y la base de "
                    "datos SQLite cuando el notebook lo solicite "
                    "(`files.upload()`), y ejecuta `insertar_cancion(...)` "
                    "para poblar la base de datos (este paso puede tardar "
                    "varios minutos porque consulta la API de iTunes "
                    "canción por canción).")))
                it.add_item(NoEscape(formatear_texto(
                    "Ejecuta las celdas donde se definen las funciones "
                    "(`retorna_metacancion`, `query_base_random`, "
                    "`mostrar_cancion`, `buscador_por_termino`, "
                    "`guardar_artista`, `preguntas_gustos`, `main_chat`).")))
                it.add_item(NoEscape(formatear_texto(
                    "Ejecuta la última celda, que llama a `main_chat()`, e "
                    "interactúa con el chatbot respondiendo en la caja de "
                    "texto que aparece bajo la celda.")))
                it.add_item(NoEscape(formatear_texto(
                    "Para pedir una canción por artista escribe una palabra "
                    "como `artista`, `cantante`, `1` o `uno`. Para "
                    "responder preguntas de gustos escribe `pregunta`, "
                    "`gustos`, `2` o `dos`. Para salir escribe `salir`, "
                    "`cerrar`, `3` o `tres`.")))
        with doc.create(Subsection('Cómo generar este informe en PDF')):
            lista(doc, [
                "Sube este script (`generar_informe_chatmp3.py`) a Colab, "
                "idealmente junto con el CSV, la base de datos SQLite y una "
                f"carpeta `{CARPETA_CAPTURAS}` con tus propias capturas.",
                "Ejecuta en una celda: `!python generar_informe_chatmp3.py`.",
                "La primera vez tardará más porque instala LaTeX "
                "automáticamente; las siguientes ejecuciones son más "
                "rápidas.",
                f"Al finalizar se genera `{NOMBRE_PDF}.pdf` en la carpeta "
                "actual. Descárgalo con `from google.colab import files` "
                f"seguido de `files.download('{NOMBRE_PDF}.pdf')`.",
            ])
        with doc.create(Subsection('Notas y posibles problemas')):
            lista(doc, [
                "La API de iTunes puede tardar o fallar ocasionalmente; el "
                "código maneja tiempos de espera (`timeout`) y errores de "
                "red, pero conviene reintentar si una consulta falla.",
                "El reconocimiento de las respuestas del usuario se basa en "
                "palabras clave exactas (no hay corrección ortográfica ni "
                "sinónimos más allá de los ya previstos en el código).",
                "Si el entorno de ejecución de Colab se reinicia, es "
                "necesario volver a ejecutar las celdas de definición de "
                "funciones antes de llamar nuevamente a `main_chat()`.",
            ])

## 8. Construcción del documento completo y generación del PDF
Ejecuta esta celda al final para generar `Informe_ChatMP3.pdf`.

In [50]:
def construir_informe():
    print(">> Preparando datos del proyecto...")



    print(">> Generando el flujograma...")
    ruta_flujograma = generar_flujograma("flujograma_chatmp3")

    print(">> Construyendo el documento LaTeX...")
    doc = Document(documentclass="article", document_options=["11pt", "a4paper"])

    for paquete, opciones in [
        ('babel', ['spanish']), ('inputenc', ['utf8']), ('fontenc', ['T1']),
        ('geometry', ['margin=2.5cm']), ('graphicx', None), ('xcolor', None),
        ('listings', None), ('hyperref', None), ('fancyhdr', None),
        ('enumitem', None), ('tcolorbox', None),
    ]:
        doc.packages.append(Package(paquete, options=opciones) if opciones
                             else Package(paquete))
    doc.preamble.append(NoEscape(r'\tcbuselibrary{skins,breakable}'))
    doc.preamble.append(NoEscape(r'''
\lstset{
  basicstyle=\ttfamily\footnotesize,
  breaklines=true,
  frame=single,
  columns=fullflexible,
  keywordstyle=\color{blue!70!black},
  commentstyle=\color{gray},
  stringstyle=\color{teal},
  showstringspaces=false,
  extendedchars=true,
  literate=%
   {á}{{\'a}}1 {é}{{\'e}}1 {í}{{\'i}}1 {ó}{{\'o}}1 {ú}{{\'u}}1%
   {Á}{{\'A}}1 {É}{{\'E}}1 {Í}{{\'I}}1 {Ó}{{\'O}}1 {Ú}{{\'U}}1%
   {ñ}{{\~n}}1 {Ñ}{{\~N}}1 {ü}{{\"u}}1 {Ü}{{\"U}}1%
   {¿}{{?`}}1 {¡}{{!`}}1%
}
'''))


    construir_portada(doc)


    construir_introduccion(doc)
    construir_problema(doc)
    construir_algoritmo(doc)
    construir_flujograma(doc, ruta_flujograma)
    construir_explicacion_codigo(doc)
    construir_instrucciones(doc)

    print(">> Compilando el PDF (esto puede tardar unos segundos)...")
    doc.generate_pdf(NOMBRE_PDF, clean_tex=False, compiler='pdflatex')
    print(f"\n>> ¡Listo! Se generó el archivo '{NOMBRE_PDF}.pdf' en la carpeta actual.")


construir_informe()

>> Preparando datos del proyecto...
>> Generando el flujograma...
>> Construyendo el documento LaTeX...
>> Compilando el PDF (esto puede tardar unos segundos)...

>> ¡Listo! Se generó el archivo 'Informe_ChatMP3.pdf' en la carpeta actual.


## 9. Descargar el PDF (solo en Google Colab)

Si estás en Google Colab, ejecuta la siguiente celda para descargar el
PDF generado. En Jupyter local no es necesaria: el archivo ya está en la
carpeta del notebook.

In [51]:
try:
    from google.colab import files
    files.download(f"{NOMBRE_PDF}.pdf")
except ImportError:
    print("No estás en Google Colab: el PDF ya quedó guardado en la carpeta actual.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>